### Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, StringType
from pyspark.sql.functions import trim, col, length

In [0]:
# Data Dictionary
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

### Reading Bronze data - sales details

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

  ###Silver Transformations

 ##Trimmings

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))


  ##Cleaning Dates

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when((col("sls_order_dt") == 0) | (length(col("sls_order_dt")) !=8), None)
        .otherwise(F.to_date(col("sls_order_dt").cast("string"),"yyyymmdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when((col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) !=8), None)
        .otherwise(F.to_date(col("sls_ship_dt").cast("string"),"yyyymmdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when((col("sls_due_dt") == 0) | (length(col("sls_due_dt")) !=8), None)
        .otherwise(F.to_date(col("sls_due_dt").cast("string"),"yyyymmdd"))
    )
)
 

  ###Sales Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when((col("sls_price").isNull()) | (col("sls_price") <= 0),
               F.when(col("sls_quantity") !=0,
               col("sls_sales")/col("sls_quantity")).otherwise(None)
    ).otherwise(col("sls_price"))
   )
)

  ##Renaming Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)


 ###Writing to Silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

  ##Sanity Checks - SQL

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 5